In [1]:
import os
from pathlib import Path
import json
import cv2 as cv
import supervision as sv
from tqdm import tqdm
import sys


In [24]:
sys.path.insert(0, "../../")
from config import MEDIA_PATH, TEMP_PATH

sys.path.insert(0, "../../packages/python")
from models import cell_segmentation as segmentators

# --- CONFIGURATION ---
# Path to the directory containing the source images
IMAGES_DIR = Path("/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/onion_cell_merged/images/train")

# Path to the COCO JSON annotation file
ANNOTATIONS_PATH = Path("/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/onion_cell_merged/images/annotations_coco_train.json")

# Path to the directory where cropped images will be saved
OUTPUT_DIR = Path("/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/onion_cell_merged/cropped_images/")

IMG_TARGET_SIDE = 200

JSON_PATH = os.path.join(TEMP_PATH, 'datasets_area_data.json')
# ---------------------

# A quick check to make sure the input paths exist before we start
assert IMAGES_DIR.exists(), f"Images directory not found at: {IMAGES_DIR}"
assert ANNOTATIONS_PATH.exists(), f"Annotations file not found at: {ANNOTATIONS_PATH}"


In [25]:
with open(JSON_PATH, 'r') as f: #json with the information of the filename of the images
    area_data = json.load(f)

resize_factor = IMG_TARGET_SIDE/area_data['INA']['lado_cuadrado']

In [26]:
def crop_and_save_detections(images_dir: Path, annotations_path: Path, output_dir: Path) -> None:
    """
    Loads COCO annotations, crops the detected objects from images, and saves
    them into class-specific folders.

    Args:
        images_dir (Path): The path to the directory containing the images.
        annotations_path (Path): The path to the COCO JSON annotation file.
        output_dir (Path): The path to the directory where cropped images will be saved.
    """
    # Ensure the main output directory exists
    output_dir.mkdir(parents=True, exist_ok=True)

    # Load the dataset using supervision
    print("Loading dataset...")
    dataset = sv.DetectionDataset.from_coco(
        images_directory_path=str(images_dir),
        annotations_path=str(annotations_path),
    )

    print(f"Found {len(dataset.classes)} classes: {dataset.classes}")

    # Iterate through the dataset with a progress bar
    for image_path, image, detections in tqdm(dataset):
        image_name = Path(image_path).stem
        image_group = image_name[0]
        image_side = area_data[image_group]['lado_cuadrado']
        image_resize_factor = int(resize_factor * image_side)

        # Iterate through each detection in the image
        for i, detection in enumerate(detections):
            # The detection object contains xyxy, mask, confidence, class_id, etc.
            xyxy, _, _, class_id, _, _ = detection

            # Get the class name for the current detection
            class_name = dataset.classes[class_id]

            # Create a directory for the class if it doesn't exist
            class_dir = output_dir / class_name
            class_dir.mkdir(parents=True, exist_ok=True)

            # Crop the detection from the image using its bounding box
            x1, y1, x2, y2 = map(int, xyxy)
            cropped_image = image[y1:y2, x1:x2]

            x, y, w, h = segmentators.CellMaskGenerator.adjust_bbox(segmentators.CellMaskGenerator, x1, y1, x2-x1, y2-y1, image_resize_factor*image_resize_factor, image.shape[1], image.shape[0])
            cropped_image = cv.resize(image[y:y+h, x:x+w], (IMG_TARGET_SIDE, IMG_TARGET_SIDE))

            # Ensure the cropped image is not empty before saving
            if cropped_image.size == 0:
                print(f"  - Skipping empty crop for detection {i} in {image_name}.png")
                continue

            # Generate a unique filename for the cropped image
            cropped_image_filename = f"{image_name}_{i}.png"
            cropped_image_path = class_dir / cropped_image_filename

            # Save the cropped image
            cv.imwrite(str(cropped_image_path), cropped_image)

    print(f"\n✅ Processing complete. Cropped images are saved in '{output_dir}'.")


In [27]:
crop_and_save_detections(
    images_dir=IMAGES_DIR,
    annotations_path=ANNOTATIONS_PATH,
    output_dir=OUTPUT_DIR,
)


Loading dataset...
Found 3 classes: ['onion-cell-dpcV-nAvS', 'd', 'nd']


100%|██████████| 452/452 [00:16<00:00, 27.34it/s]


✅ Processing complete. Cropped images are saved in '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/onion_cell_merged/cropped_images'.
